# Issues #9–11 — run LLaMA 3.2 / Mistral 7B on all tasks (GPU)

Runs the local HF models through the same harness as the API models.
**Set `CONDITION` below** (`raw` for #9, `segmented` for #10, `prompt_guided` for #11) and run all cells.

Fully resumable: results + response cache live on Drive, so a VM recycle
loses nothing — just rerun the run cell.

Runtime: **T4 GPU** works (Mistral-7B in fp16 just fits); L4/A100 is faster.
Expect roughly 1–2 h per model per condition.

In [ ]:
# 1. GPU check
import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Mount Drive (persistent cache + results)
from google.colab import drive
drive.mount('/content/drive')
import os
PERSIST = '/content/drive/MyDrive/nlp_final_runs'
os.makedirs(PERSIST, exist_ok=True)

In [ ]:
# 3. Clone the repo (private -> needs PAT) and install deps
from getpass import getpass
import os, subprocess
if not os.path.exists('/content/NLP-Final-'):
    pat = getpass('GitHub PAT: ')
    subprocess.run(['git', 'clone',
        f'https://{pat}@github.com/AdonZahavi/NLP-Final-.git',
        '/content/NLP-Final-'], check=True)
%cd /content/NLP-Final-
!pip -q install transformers accelerate sentencepiece python-dotenv
!pip -q install -U datasets

In [ ]:
# 4. HF token (LLaMA is gated) + link Drive-backed cache & results
from huggingface_hub import login
login()  # paste HF token with access to meta-llama/Llama-3.2-3B-Instruct

import os
# response cache and results dirs -> symlinks into Drive so they survive recycles
for name in ('cache', 'results'):
    target = f'{PERSIST}/{name}'
    os.makedirs(target, exist_ok=True)
    if os.path.islink(name):
        os.unlink(name)
    elif os.path.isdir(name):
        import shutil; shutil.rmtree(name)
    os.symlink(target, name)
print('cache ->', os.path.realpath('cache'))
print('results ->', os.path.realpath('results'))

In [ ]:
# 5. Configuration
CONDITION = 'raw'   # <- 'raw' (#9) | 'segmented' (#10) | 'prompt_guided' (#11)
MODELS = ['llama-3.2', 'mistral-7b']   # run both, LLaMA first (smaller)
SMOKE_LIMIT = 0     # set 20 for a smoke test, 0 for the full run

In [ ]:
# 6. Smoke + full run (resumable — rerun this cell after any crash)
import subprocess, sys
for model in MODELS:
    cmd = [sys.executable, 'scripts/run_all.py',
           '--condition', CONDITION, '--models', model]
    if SMOKE_LIMIT:
        cmd += ['--limit', str(SMOKE_LIMIT)]
    print('\n=== ', ' '.join(cmd), ' ===')
    r = subprocess.run(cmd, env={**__import__('os').environ, 'PYTHONPATH': 'src'})
    if r.returncode != 0:
        print(f'!! {model} exited {r.returncode} — rerun this cell to resume')

In [ ]:
# 7. Sanity report
!PYTHONPATH=src python scripts/summarize_results.py --condition {CONDITION}

In [ ]:
# 8. Copy results back into the repo tree and push
# (results/ is a symlink to Drive; copy real files into the repo for commit)
import shutil, os, subprocess
os.makedirs('/content/push_tmp', exist_ok=True)
subprocess.run(['git', 'config', 'user.email', 'orna.zahavi1@gmail.com'], check=True)
subprocess.run(['git', 'config', 'user.name', 'AdonZahavi'], check=True)
os.unlink('results')
shutil.copytree(f'{PERSIST}/results', 'results')
!git add results/ && git status --short results/ | head
BRANCH = f'issue-9-hf-runs-{CONDITION}'
!git checkout -b {BRANCH} 2>/dev/null || git checkout {BRANCH}
!git commit -m "HF model results: {CONDITION} condition (llama-3.2, mistral-7b)"
!git push -u origin {BRANCH}